# FBS Detection — Random Forest

Random Forest model for False Base Station detection. Part of the AI model benchmark — see `fbs_model_benchmark.ipynb` for comparative analysis of training overhead and detection performance.

In [ ]:
import json
import time
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_curve,
    roc_auc_score,
    accuracy_score,
    matthews_corrcoef,
)
import matplotlib.pyplot as plt
import seaborn as sns

## 1. Load Preprocessed Splits

Splits are generated by `fbs_dataset_analysis.ipynb` — run that notebook first to produce `fbs_train.csv`, `fbs_test.csv`, and `fbs_class_weights.json`.

In [ ]:
TRAIN_PATH = Path("fbs_train.csv")
TEST_PATH  = Path("fbs_test.csv")
CW_PATH    = Path("fbs_class_weights.json")

for p in [TRAIN_PATH, TEST_PATH]:
    if not p.exists():
        raise FileNotFoundError(
            f"Split file not found: {p}\n"
            "Run fbs_dataset_analysis.ipynb first to generate all split files."
        )

train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

feature_cols = [c for c in train_df.columns if c not in ("label", "source")]

# Use numpy arrays for Random Forest
X_train = train_df[feature_cols]
y_train = train_df["label"].values
X_test  = test_df[feature_cols]
y_test  = test_df["label"].values

if CW_PATH.exists():
    with open(CW_PATH) as f:
        cw_dict = {int(k): v for k, v in json.load(f).items()}
    print(f"Class weights loaded: {cw_dict}")
else:
    cw_dict = "balanced"
    print("fbs_class_weights.json not found — falling back to class_weight='balanced'")

print(f"\nTrain : {len(train_df):,} rows  (normal={int((y_train==0).sum()):,}, attack={int((y_train==1).sum()):,})")
print(f"Test  : {len(test_df):,} rows  (normal={int((y_test==0).sum()):,}, attack={int((y_test==1).sum()):,})")
print(f"Features: {len(feature_cols)}")

## 2. Data Ready

Feature selection, presence-indicator engineering, median imputation, and stratified 70/30 splitting were all performed in `fbs_dataset_analysis.ipynb`. Arrays `X_train`, `X_test` and labels are ready for scaling and training.

In [ ]:
print(f"Train shape : {X_train.shape}")
print(f"Test shape  : {X_test.shape}")

In [ ]:
print(f"Train : {X_train.shape}  |  Test : {X_test.shape}")
print(f"Features: {len(feature_cols)}")

## 3. Random Forest

Random Forest for False Base Station detection. Part of the AI model benchmark — see `fbs_model_benchmark.ipynb` for comparative analysis of training overhead and detection performance.

In [ ]:
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=3,
    max_features="sqrt",  # ~27 of 762 features per split; reduces overfitting on high-dim data
    class_weight=cw_dict if isinstance(cw_dict, dict) else "balanced",
    random_state=42,
    n_jobs=-1,
)

t0 = time.perf_counter()
model.fit(X_train, y_train)
train_time_sec = time.perf_counter() - t0
print(f"Random Forest training complete in {train_time_sec:.2f}s")

In [ ]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = cross_validate(
    model, X_train, y_train,
    cv=cv,
    scoring=["accuracy", "f1_macro", "roc_auc"],
    n_jobs=-1,
)

print("5-Fold Cross-Validation (on training set):")
print(f"  Accuracy  : {cv_results['test_accuracy'].mean():.4f} ± {cv_results['test_accuracy'].std():.4f}")
print(f"  F1 Macro  : {cv_results['test_f1_macro'].mean():.4f} ± {cv_results['test_f1_macro'].std():.4f}")
print(f"  ROC-AUC   : {cv_results['test_roc_auc'].mean():.4f} ± {cv_results['test_roc_auc'].std():.4f}")

In [ ]:
X_train_final = X_train
X_test_final  = X_test

print(f"Features used : {X_train_final.shape[1]}")
print(f"Train rows    : {X_train_final.shape[0]}")
print(f"Test  rows    : {X_test_final.shape[0]}")

## 6. Evaluation and Visualization

In [ ]:
n_test = len(X_test_final)
t0 = time.perf_counter()
y_test_pred  = model.predict(X_test_final)
inference_sec = time.perf_counter() - t0
ms_per_sample = (inference_sec / n_test) * 1000
y_test_proba = model.predict_proba(X_test_final)[:, 1]

acc = accuracy_score(y_test, y_test_pred)
mcc = matthews_corrcoef(y_test, y_test_pred)
auc = roc_auc_score(y_test, y_test_proba)

print("=" * 55)
print("Test Set (Final)")
print("=" * 55)
print(f"  Accuracy : {acc:.4f}")
print(f"  MCC      : {mcc:.4f}  (1.0 = perfect, 0 = random)")
print(f"  ROC-AUC  : {auc:.4f}")
print()
print(classification_report(y_test, y_test_pred, target_names=["Normal", "Attack"]))

In [ ]:
cm = confusion_matrix(y_test, y_test_pred)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Normal", "Attack"],
            yticklabels=["Normal", "Attack"], ax=ax)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix — Test Set")
plt.tight_layout()
plt.show()

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_test_proba)

fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(fpr, tpr, label=f"Random Forest (AUC = {auc:.4f})")
ax.plot([0, 1], [0, 1], "k--", label="Random")
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve — Test Set")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Random Forest feature importances (Gini importance)
importance = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)

n_top = min(25, len(feature_cols))
fig, ax = plt.subplots(figsize=(10, 8))
importance.head(n_top).sort_values().plot(kind="barh", ax=ax)
ax.set_xlabel("Feature Importance (Gini)")
ax.set_title(f"Top {n_top} Features — Random Forest")
plt.tight_layout()
plt.show()

print(f"\nTop 10 features:")
print(importance.head(10).to_string())

In [ ]:
# ── Per-attack-type accuracy breakdown ────────────────────────────────────────
# Identifies which attack types the model handles well vs misses entirely.
# With ~9 training samples per type, some types are almost certainly weak spots.

if "source" in test_df.columns:
    results = test_df[["label", "source"]].copy()
    results["predicted"] = y_test_pred
    results["correct"]   = (results["predicted"] == results["label"]).astype(int)

    summary = (
        results.groupby(["label", "source"])
        .agg(n=("correct", "count"), acc=("correct", "mean"))
        .reset_index()
        .sort_values(["label", "acc"])
    )

    print(f"{'Source':<52}  {'Class':6}  {'N':>4}  {'Acc':>6}")
    print("-" * 74)
    for _, row in summary.iterrows():
        label_str = "normal" if row["label"] == 0 else "attack"
        flag      = "  ⚠ low" if row["acc"] < 0.70 else ""
        print(f"  {row['source']:<52}  {label_str:6}  {int(row['n']):>4}  {row['acc']:>6.3f}{flag}")
else:
    print("No 'source' column in test_df — re-run fbs_dataset_analysis.ipynb to regenerate splits.")

In [ ]:
# Export metrics for fbs_model_benchmark.ipynb comparative analysis
from sklearn.metrics import precision_score, recall_score, f1_score

metrics = {
    "model": "Random Forest",
    "train_time_sec": train_time_sec,
    "inference_batch_sec": inference_sec,
    "inference_ms_per_sample": ms_per_sample,
    "accuracy": acc,
    "precision_macro": precision_score(y_test, y_test_pred, average="macro", zero_division=0),
    "recall_macro": recall_score(y_test, y_test_pred, average="macro", zero_division=0),
    "f1_macro": f1_score(y_test, y_test_pred, average="macro", zero_division=0),
    "roc_auc": auc,
    "mcc": mcc,
}

with open("fbs_benchmark_rf.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("Saved fbs_benchmark_rf.json")